In [ ]:
import os
import copy
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.utils import dropout_edge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

# Paths
DATASET_PATH = "your path"
RESULT_PATH = "your path"
os.makedirs(RESULT_PATH, exist_ok=True)

# Load dataset
df = pd.read_csv(DATASET_PATH)
df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y')
df = df.sort_values(['Company', 'Date']).reset_index(drop=True)

# Normalize features
features = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
scalers = {}
for feat in features:
    scaler = StandardScaler()
    df[feat] = scaler.fit_transform(df[[feat]])
    scalers[feat] = scaler

# Define prediction target: next day Close price
df['Target_Close'] = df.groupby('Company')['Close'].shift(-1)
df = df.dropna(subset=['Target_Close']).reset_index(drop=True)

# Create graphs per company per month
def create_graphs(df):
    graphs = []
    df['YearMonth'] = df['Date'].dt.to_period('M')
    grouped = df.groupby(['Company', 'YearMonth'])

    for (company, ym), group in grouped:
        group = group.sort_values('Date').reset_index(drop=True)
        if len(group) < 2:
            continue
        x = torch.tensor(group[features].values, dtype=torch.float)
        y = torch.tensor([group['Target_Close'].values[-1]], dtype=torch.float).view(-1, 1)

        edge_index = []
        for i in range(len(group) - 1):
            edge_index.append([i, i+1])
            edge_index.append([i+1, i])
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

        data = Data(x=x, edge_index=edge_index, y=y)
        data.company = company
        data.year_month = str(ym)
        graphs.append(data)
    return graphs

graphs = create_graphs(df)
print(f"Total graphs created: {len(graphs)}")

# Split graphs by company (simulate clients)
clients = {}
for g in graphs:
    clients.setdefault(g.company, []).append(g)
print(f"Clients: {list(clients.keys())}")

# Model with contrastive learning support
class GCNContrastive(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(GCNContrastive, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.lin = torch.nn.Linear(hidden_channels, 1)

    def encode(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

    def forward(self, x, edge_index, batch):
        node_emb = self.encode(x, edge_index)
        graph_emb = global_mean_pool(node_emb, batch)
        out = self.lin(graph_emb)
        return out, graph_emb

# Graph augmentation: edge dropout
def graph_augmentation(data, drop_edge_prob=0.2):
    edge_index, _ = dropout_edge(data.edge_index, p=drop_edge_prob)
    data_aug = Data(x=data.x, edge_index=edge_index, y=data.y)
    
    if hasattr(data, 'batch'):
        data_aug.batch = data.batch
    else:
      
        data_aug.batch = torch.zeros(data.x.size(0), dtype=torch.long)
    return data_aug


# Contrastive loss (InfoNCE)
def contrastive_loss(z1, z2, temperature=0.5):
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    batch_size = z1.size(0)

    representations = torch.cat([z1, z2], dim=0)
    similarity_matrix = torch.matmul(representations, representations.t())

    mask = (~torch.eye(2 * batch_size, 2 * batch_size, dtype=bool)).float().to(z1.device)
    positives = torch.cat([torch.diag(similarity_matrix, batch_size),
                           torch.diag(similarity_matrix, -batch_size)], dim=0)
    nominator = torch.exp(positives / temperature)
    denominator = mask * torch.exp(similarity_matrix / temperature)
    loss = -torch.log(nominator / denominator.sum(dim=1))
    return loss.mean()

# Contrastive training function
def train_contrastive(model, loader, optimizer, device, contrastive_weight=0.1):
    model.train()
    total_loss = 0
    for data in loader:
        data = data.to(device)
        data1 = graph_augmentation(data)
        data2 = graph_augmentation(data)
        data1 = data1.to(device)
        data2 = data2.to(device)

        optimizer.zero_grad()
        out1, z1 = model(data1.x, data1.edge_index, data1.batch.to(device))
        out2, z2 = model(data2.x, data2.edge_index, data2.batch.to(device))

        mse_loss = (F.mse_loss(out1, data.y) + F.mse_loss(out2, data.y)) / 2
        c_loss = contrastive_loss(z1, z2)
        loss = mse_loss + contrastive_weight * c_loss

        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
    return total_loss / len(loader.dataset)

# Evaluation function
def evaluate(model, loader, device):
    model.eval()
    preds = []
    trues = []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out, _ = model(data.x, data.edge_index, data.batch.to(device))
            preds.append(out.cpu())
            trues.append(data.y.cpu())
    preds = torch.cat(preds).view(-1)
    trues = torch.cat(trues).view(-1)
    mse = mean_squared_error(trues, preds)
    return mse, preds, trues

from torch_geometric.loader import DataLoader

# Prepare loaders per client
client_loaders = {}
for c, g_list in clients.items():
    n = len(g_list)
    train_n = int(0.8 * n)
    train_loader = DataLoader(g_list[:train_n], batch_size=4, shuffle=True)
    test_loader = DataLoader(g_list[train_n:], batch_size=4, shuffle=False)
    client_loaders[c] = (train_loader, test_loader)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
in_channels = len(features)
hidden_channels = 64

def get_model():
    model = GCNContrastive(in_channels, hidden_channels)
    return model.to(device)

def average_weights(w_list):
    avg_w = copy.deepcopy(w_list[0])
    for key in avg_w.keys():
        for w in w_list[1:]:
            avg_w[key] += w[key]
        avg_w[key] = torch.div(avg_w[key], len(w_list))
    return avg_w

epochs_per_round = 3
rounds = 10

global_model = get_model()
global_weights = global_model.state_dict()

mse_per_round = []

for r in range(rounds):
    print(f"Round {r+1}/{rounds}")
    local_weights = []
    for c, (train_loader, _) in client_loaders.items():
        local_model = get_model()
        local_model.load_state_dict(global_weights)
        optimizer = torch.optim.Adam(local_model.parameters(), lr=0.01)
        for e in range(epochs_per_round):
            loss = train_contrastive(local_model, train_loader, optimizer, device, contrastive_weight=0.1)
            print(f"Client {c} Epoch {e+1} Loss: {loss:.4f}")
        local_weights.append(local_model.state_dict())
    global_weights = average_weights(local_weights)
    global_model.load_state_dict(global_weights)

    # Evaluate on all clients combined test set
    all_test_data = []
    for _, (_, test_loader) in client_loaders.items():
        all_test_data += test_loader.dataset
    all_test_loader = DataLoader(all_test_data, batch_size=8, shuffle=False)
    mse, preds, trues = evaluate(global_model, all_test_loader, device)
    print(f"Global Model MSE after round {r+1}: {mse:.4f}")
    mse_per_round.append(mse)

# Save plot and MSEs
plt.figure()
plt.plot(range(1, rounds+1), mse_per_round, marker='o')
plt.title("Global Model MSE per FL Round with Contrastive Learning")
plt.xlabel("Round")
plt.ylabel("MSE")
plt.grid(True)
plt.savefig(os.path.join(RESULT_PATH, "mse_per_round_contrastive.png"))
plt.close()

with open(os.path.join(RESULT_PATH, "mse_per_round_contrastive.txt"), "w") as f:
    for i, val in enumerate(mse_per_round, 1):
        f.write(f"Round {i}: MSE={val}\n")

print(f"Training finished. Results saved in {RESULT_PATH}")


Total graphs created: 2636
Clients: ['DELL', 'IBM', 'INTC', 'MSFT', 'SONY', 'VZ']
Round 1/10
Client DELL Epoch 1 Loss: 0.1434
Client DELL Epoch 2 Loss: 0.1327
Client DELL Epoch 3 Loss: 0.1203
Client IBM Epoch 1 Loss: 0.1863
Client IBM Epoch 2 Loss: 0.1085
Client IBM Epoch 3 Loss: 0.1128
Client INTC Epoch 1 Loss: 0.1396
Client INTC Epoch 2 Loss: 0.1213
Client INTC Epoch 3 Loss: 0.1013
Client MSFT Epoch 1 Loss: 0.1306
Client MSFT Epoch 2 Loss: 0.0998
Client MSFT Epoch 3 Loss: 0.1085
Client SONY Epoch 1 Loss: 0.1783
Client SONY Epoch 2 Loss: 0.1319
Client SONY Epoch 3 Loss: 0.1356
Client VZ Epoch 1 Loss: 0.1127
Client VZ Epoch 2 Loss: 0.0999
Client VZ Epoch 3 Loss: 0.0977
Global Model MSE after round 1: 0.0418
Round 2/10
Client DELL Epoch 1 Loss: 0.1063
Client DELL Epoch 2 Loss: 0.1045
Client DELL Epoch 3 Loss: 0.1074
Client IBM Epoch 1 Loss: 0.1713
Client IBM Epoch 2 Loss: 0.1173
Client IBM Epoch 3 Loss: 0.1068
Client INTC Epoch 1 Loss: 0.1063
Client INTC Epoch 2 Loss: 0.0962
Client INTC